In [10]:
import subprocess
import sys
import os
import json
import numpy as np
import pandas as pd
import joblib
import re
import warnings
from urllib.parse import urlparse
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, classification_report
)

warnings.filterwarnings('ignore')

print("Phase 8 — Testing, Validation & Documentation")
print("=" * 50)

Phase 8 — Testing, Validation & Documentation


In [11]:
import subprocess, sys, os

test_dir  = os.path.abspath('../tests')
test_file = os.path.join(test_dir, 'test_pipeline.py')
os.makedirs(test_dir, exist_ok=True)

# Always recreate — ensures correct values
if os.path.exists(test_file):
    os.remove(test_file)
    print("Old test_pipeline.py removed")

test_code = '''"""
test_pipeline.py
Automated test suite for Phishing Detection Framework.
Label convention: 0.0 = Phishing  |  1.0 = Legitimate
Run with: pytest tests/test_pipeline.py -v
"""

import pytest
import numpy as np
import pandas as pd
import joblib
import json
import os
from sklearn.metrics import f1_score

BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

def path(rel):
    return os.path.join(BASE, rel)


class TestModelsLoad:

    def test_random_forest_loads(self):
        model = joblib.load(path("models/random_forest.pkl"))
        assert model is not None
        assert hasattr(model, "predict")

    def test_svm_loads(self):
        svm = joblib.load(path("models/svm_model.pkl"))
        assert svm is not None
        assert hasattr(svm, "predict")

    def test_scaler_loads(self):
        scaler = joblib.load(path("models/scaler.pkl"))
        assert hasattr(scaler, "transform")
        assert scaler.n_features_in_ == 16, \\
            f"Scaler expects 16 features, got {scaler.n_features_in_}"

    def test_isolation_forest_loads(self):
        iso = joblib.load(path("models/isolation_forest.pkl"))
        assert iso is not None
        assert hasattr(iso, "predict")


class TestDataFiles:

    def test_feature_cols_exists(self):
        feat = np.load(path("data/processed/feature_cols.npy"),
                       allow_pickle=True).tolist()
        assert len(feat) == 16, \\
            f"Expected 16 features, found {len(feat)}"

    def test_test_arrays_shape(self):
        X = np.load(path("data/processed/X_test_scaled.npy"))
        y = np.load(path("data/processed/y_test.npy"))
        assert X.shape[1] == 16, \\
            f"X_test has {X.shape[1]} cols, expected 16"
        assert len(X) == len(y)

    def test_log_features_scored_exists(self):
        df = pd.read_csv(
            path("data/processed/log_features_scored.csv")
        )
        assert "anomaly_label" in df.columns
        assert len(df) > 0

    def test_graph_file_exists(self):
        assert os.path.exists(path("models/phishing_graph.gml"))


class TestPredictions:

    def setup_method(self):
        self.model  = joblib.load(path("models/random_forest.pkl"))
        self.X_test = np.load(path("data/processed/X_test_scaled.npy"))
        self.y_test = np.load(path("data/processed/y_test.npy"))

    def test_prediction_returns_binary(self):
        preds = self.model.predict(self.X_test[:50])
        assert set(preds).issubset({0.0, 1.0})

    def test_prediction_count_matches(self):
        preds = self.model.predict(self.X_test[:100])
        assert len(preds) == 100

    def test_probability_valid(self):
        probas = self.model.predict_proba(self.X_test[:20])
        assert probas.shape[1] == 2
        assert np.allclose(probas.sum(axis=1), 1.0, atol=1e-6)


class TestModelPerformance:

    def setup_method(self):
        self.model  = joblib.load(path("models/random_forest.pkl"))
        self.X_test = np.load(path("data/processed/X_test_scaled.npy"))
        self.y_test = np.load(path("data/processed/y_test.npy"))

    def test_f1_above_threshold(self):
        preds = self.model.predict(self.X_test)
        f1    = f1_score(self.y_test, preds, average="macro")
        assert f1 >= 0.90, f"F1 {f1:.4f} below 0.90"


class TestLogAnomalyDetection:

    def setup_method(self):
        self.df = pd.read_csv(
            path("data/processed/log_features_scored.csv")
        )

    def test_anomaly_column_exists(self):
        assert "anomaly_label" in self.df.columns

    def test_anomaly_labels_valid(self):
        valid = {"Normal", "ANOMALY"}
        assert set(self.df["anomaly_label"].unique()).issubset(valid)

    def test_known_attackers_detected(self):
        known = ["10.0.0.99", "185.220.101.45", "172.16.5.200"]
        for ip in known:
            rows = self.df[self.df["ip"] == ip]
            assert len(rows) > 0, f"{ip} not found"
            assert rows.iloc[0]["anomaly_label"] == "ANOMALY", \\
                f"{ip} not detected"


class TestAdaptiveRetraining:

    def setup_method(self):
        log_path = path("reports/retraining_log.json")
        assert os.path.exists(log_path)
        with open(log_path) as f:
            self.log = json.load(f)

    def test_log_has_entries(self):
        assert len(self.log) >= 1

    def test_log_required_fields(self):
        required = ["timestamp", "f1_before", "retrained", "threshold"]
        for entry in self.log:
            for field in required:
                assert field in entry

    def test_f1_values_valid(self):
        for entry in self.log:
            assert 0.0 <= entry["f1_before"] <= 1.0

    def test_thresholds_valid(self):
        thresholds = [e["threshold"] for e in self.log]
        assert all(0.0 < t <= 1.0 for t in thresholds), \\
            "Invalid threshold values found in log"
'''

with open(test_file, 'w', encoding='utf-8') as f:
    f.write(test_code)
print(f"Created: {test_file}")

# Run pytest
print("\nRunning automated test suite...\n")
result = subprocess.run(
    [sys.executable, '-m', 'pytest',
     'tests/test_pipeline.py',
     '-v', '--tb=short', '--no-header'],
    capture_output=True,
    text=True,
    cwd=os.path.abspath('../')
)

print(result.stdout)
if result.returncode != 0 and result.stderr:
    print("STDERR:", result.stderr[:500])

lines  = result.stdout.split('\n')
passed = sum(1 for l in lines if 'PASSED' in l)
failed = sum(1 for l in lines if 'FAILED' in l)
errors = sum(1 for l in lines if 'ERROR'  in l)

print(f"\n{'='*45}")
print(f"PYTEST RESULTS")
print(f"{'='*45}")
print(f"  Passed : {passed}")
print(f"  Failed : {failed}")
print(f"  Errors : {errors}")
print(f"  Status : {'ALL PASSED' if failed==0 and errors==0 else 'SOME FAILED'}")
print(f"{'='*45}")

Created: c:\Users\shubh\phishing-detection-framework\tests\test_pipeline.py

Running automated test suite...

============================= test session starts =============================
collecting ... collected 19 items

tests/test_pipeline.py::TestModelsLoad::test_random_forest_loads PASSED  [  5%]
tests/test_pipeline.py::TestModelsLoad::test_svm_loads PASSED            [ 10%]
tests/test_pipeline.py::TestModelsLoad::test_scaler_loads PASSED         [ 15%]
tests/test_pipeline.py::TestModelsLoad::test_isolation_forest_loads PASSED [ 21%]
tests/test_pipeline.py::TestDataFiles::test_feature_cols_exists PASSED   [ 26%]
tests/test_pipeline.py::TestDataFiles::test_test_arrays_shape PASSED     [ 31%]
tests/test_pipeline.py::TestDataFiles::test_log_features_scored_exists PASSED [ 36%]
tests/test_pipeline.py::TestDataFiles::test_graph_file_exists PASSED     [ 42%]
tests/test_pipeline.py::TestPredictions::test_prediction_returns_binary PASSED [ 47%]
tests/test_pipeline.py::TestPredictions::t

In [7]:
model  = joblib.load('../models/random_forest.pkl')
scaler = joblib.load('../models/scaler.pkl')

# Use base 16 features — matches scaler and model
feature_cols = np.load(
    '../data/processed/feature_cols.npy',   # FIXED: was feature_cols_with_graph.npy
    allow_pickle=True
).tolist()

print(f"Model features : {model.n_features_in_}")
print(f"Scaler features: {scaler.n_features_in_}")
print(f"Feature list   : {len(feature_cols)}")
print(f"Label convention: 0.0=Phishing  |  1.0=Legitimate")


def extract_features(url):
    try:
        parsed = urlparse(url)
        domain = parsed.netloc
        return {
            'url_length'         : len(url),
            'has_ip_address'     : 1 if re.match(
                                   r'\d+\.\d+\.\d+\.\d+',
                                   domain) else 0,
            'dot_count'          : url.count('.'),
            'https_flag'         : 1 if url.startswith('https') else 0,
            'url_entropy'        : 0,
            'token_count'        : len(re.split(r'[/\-_?=&.]', url)),
            'subdomain_count'    : max(len(domain.split('.')) - 2, 0),
            'query_param_count'  : url.count('='),
            'tld_length'         : len(domain.split('.')[-1])
                                   if '.' in domain else 0,
            'path_length'        : len(parsed.path),
            'has_hyphen_in_domain': 1 if '-' in domain else 0,
            'number_of_digits'   : sum(c.isdigit() for c in url),
            'tld_popularity'     : 0,
            'suspicious_file_extension': 1 if any(
                url.endswith(e) for e in
                ['.exe','.php','.js','.sh','.bat']
            ) else 0,
            'domain_name_length' : len(domain),
            'percentage_numeric_chars': (
                sum(c.isdigit() for c in url) / max(len(url), 1)
            ),
        }
    except:
        return None


def predict_url(url):
    feats = extract_features(url)
    if feats is None:
        return None, None
    # Align to exact training feature order
    row    = {col: feats.get(col, 0) for col in feature_cols}
    df_row = pd.DataFrame([row])[feature_cols]
    scaled = scaler.transform(df_row)
    pred   = model.predict(scaled)[0]
    proba  = model.predict_proba(scaled)[0]

    # 0.0=Phishing, 1.0=Legitimate
    # proba[0]=prob of class 0.0 (Phishing)
    # proba[1]=prob of class 1.0 (Legitimate)
    if pred == 0.0:
        label = 'Phishing'
        conf  = proba[0] * 100   # confidence it is phishing
    else:
        label = 'Legitimate'
        conf  = proba[1] * 100   # confidence it is legitimate
    return label, conf


# Adversarial phishing URLs — crafted to look legitimate
adversarial_phishing = [
    ("https://paypal.com.account-verify.net/login",
     "Legitimate-looking domain with extra subdomain"),
    ("https://secure-banking.com/login/verify",
     "Uses HTTPS and secure keywords"),
    ("https://google-security.com/account/confirm",
     "Impersonates Google with HTTPS"),
    ("https://amazon-orders.net/account/signin",
     "Impersonates Amazon store"),
    ("https://microsoft365-login.com/auth",
     "Impersonates Microsoft with HTTPS"),
]

# Known legitimate URLs
known_legitimate = [
    ("https://www.google.com",       "Major search engine"),
    ("https://github.com/login",     "Developer platform"),
    ("https://www.microsoft.com/en-us/", "Official Microsoft site"),
    ("https://stackoverflow.com/questions", "Technical Q&A site"),
]

print("\n" + "=" * 70)
print("ADVERSARIAL PHISHING URL TEST")
print("0.0=Phishing  |  1.0=Legitimate")
print("=" * 70)
print(f"\n{'URL':<52} {'Prediction':<12} {'Confidence'}")
print("-" * 70)

adv_detected = 0
for url, desc in adversarial_phishing:
    label, conf = predict_url(url)
    if label is None:
        print(f"{url[:50]:<52} ERROR")
        continue
    detected = "CAUGHT" if label == 'Phishing' else "MISSED"
    if label == 'Phishing':
        adv_detected += 1
    print(f"{url[:50]:<52} {label:<12} {conf:.1f}%  {detected}")
    print(f"  ({desc})")

print(f"\nAdversarial detection rate: "
      f"{adv_detected}/{len(adversarial_phishing)} "
      f"({adv_detected/len(adversarial_phishing)*100:.0f}%)")

print(f"\n{'='*70}")
print("LEGITIMATE URL TEST")
print("These should all be classified as Legitimate")
print("=" * 70)
print(f"\n{'URL':<52} {'Prediction':<12} {'Confidence'}")
print("-" * 70)

legit_correct = 0
for url, desc in known_legitimate:
    label, conf = predict_url(url)
    if label is None:
        print(f"{url[:50]:<52} ERROR")
        continue
    correct = "CORRECT" if label == 'Legitimate' else "WRONG"
    if label == 'Legitimate':
        legit_correct += 1
    print(f"{url[:50]:<52} {label:<12} {conf:.1f}%  {correct}")

print(f"\nLegitimate accuracy: "
      f"{legit_correct}/{len(known_legitimate)} "
      f"({legit_correct/len(known_legitimate)*100:.0f}%)")

Model features : 16
Scaler features: 16
Feature list   : 16
Label convention: 0.0=Phishing  |  1.0=Legitimate

ADVERSARIAL PHISHING URL TEST
0.0=Phishing  |  1.0=Legitimate

URL                                                  Prediction   Confidence
----------------------------------------------------------------------
https://paypal.com.account-verify.net/login          Legitimate   83.2%  MISSED
  (Legitimate-looking domain with extra subdomain)
https://secure-banking.com/login/verify              Legitimate   92.9%  MISSED
  (Uses HTTPS and secure keywords)
https://google-security.com/account/confirm          Legitimate   91.9%  MISSED
  (Impersonates Google with HTTPS)
https://amazon-orders.net/account/signin             Legitimate   92.5%  MISSED
  (Impersonates Amazon store)
https://microsoft365-login.com/auth                  Legitimate   87.5%  MISSED
  (Impersonates Microsoft with HTTPS)

Adversarial detection rate: 0/5 (0%)

LEGITIMATE URL TEST
These should all be classified

In [8]:
X_test = np.load('../data/processed/X_test_scaled.npy')
y_test = np.load('../data/processed/y_test.npy')

y_pred = model.predict(X_test)

# Use macro average — balanced across both classes
f1  = f1_score(y_test, y_pred, average='macro')
pre = precision_score(y_test, y_pred, average='macro')
rec = recall_score(y_test, y_pred, average='macro')
acc = accuracy_score(y_test, y_pred)

# sklearn confusion matrix unpacks as:
# tn=0pred0, fp=0pred1, fn=1pred0, tp=1pred1
# 0.0=Phishing so: tn=Phishing correctly caught
#                  fp=Phishing wrongly cleared
#                  fn=Legitimate wrongly flagged
#                  tp=Legitimate correctly cleared
from sklearn.metrics import confusion_matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print("=" * 50)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("Label convention: 0.0=Phishing  |  1.0=Legitimate")
print("=" * 50)
print(f"\nMetrics on held-out test set ({len(X_test):,} samples):")
print(f"  Accuracy      : {acc*100:.2f}%")
print(f"  Precision     : {pre:.4f}  (macro avg)")
print(f"  Recall        : {rec:.4f}  (macro avg)")
print(f"  F1 Score      : {f1:.4f}  (macro avg)")
print(f"\nConfusion Matrix:")
print(f"  Phishing correctly caught    : {tn:,}")
print(f"  Phishing missed (slipped)    : {fp:,}")
print(f"  Legitimate correctly cleared : {tp:,}")
print(f"  Legitimate wrongly flagged   : {fn:,}")
print(f"\nOperational Rates:")
print(f"  Phishing miss rate   : {fp/(fp+tn)*100:.2f}%")
print(f"  Legit false alarm    : {fn/(fn+tp)*100:.2f}%")
print(f"\n{'='*50}")
print(classification_report(
    y_test, y_pred,
    target_names=['Phishing (0.0)', 'Legitimate (1.0)']
))

FINAL MODEL PERFORMANCE SUMMARY
Label convention: 0.0=Phishing  |  1.0=Legitimate

Metrics on held-out test set (20,175 samples):
  Accuracy      : 99.58%
  Precision     : 0.9946  (macro avg)
  Recall        : 0.9965  (macro avg)
  F1 Score      : 0.9955  (macro avg)

Confusion Matrix:
  Phishing correctly caught    : 7,470
  Phishing missed (slipped)    : 6
  Legitimate correctly cleared : 12,621
  Legitimate wrongly flagged   : 78

Operational Rates:
  Phishing miss rate   : 0.08%
  Legit false alarm    : 0.61%

                  precision    recall  f1-score   support

  Phishing (0.0)       0.99      1.00      0.99      7476
Legitimate (1.0)       1.00      0.99      1.00     12699

        accuracy                           1.00     20175
       macro avg       0.99      1.00      1.00     20175
    weighted avg       1.00      1.00      1.00     20175



In [9]:
with open('../reports/retraining_log.json') as f:
    retrain_log = json.load(f)

phase8_report = {
    'phase'            : 8,
    'label_convention' : '0.0=Phishing  |  1.0=Legitimate',
    'pytest': {
        'total_tests' : passed + failed + errors,
        'passed'      : passed,
        'failed'      : failed,
        'errors'      : errors,
        'all_passed'  : failed == 0 and errors == 0,
    },
    'adversarial_testing': {
        'phishing_urls_tested'    : len(adversarial_phishing),
        'phishing_detected'       : adv_detected,
        'detection_rate_pct'      : round(
            adv_detected/len(adversarial_phishing)*100, 1),
        'legitimate_urls_tested'  : len(known_legitimate),
        'legitimate_correct'      : legit_correct,
        'legitimate_accuracy_pct' : round(
            legit_correct/len(known_legitimate)*100, 1),
        'note': (
            'Adversarial URLs use HTTPS and clean structure — '
            'URL features alone are insufficient to catch them. '
            'Graph features (domain_url_count) would improve this.'
        )
    },
    'final_model_performance': {
        'test_samples'   : int(len(X_test)),
        'accuracy'       : round(float(acc),  4),
        'precision_macro': round(float(pre),  4),
        'recall_macro'   : round(float(rec),  4),
        'f1_macro'       : round(float(f1),   4),
        'phishing_caught'  : int(tn),
        'phishing_missed'  : int(fp),
        'legit_cleared'    : int(tp),
        'legit_flagged'    : int(fn),
        'phishing_miss_pct': round(fp/(fp+tn)*100, 2),
        'legit_alarm_pct'  : round(fn/(fn+tp)*100, 2),
    },
    'adaptive_module': {
        'total_checks'     : len(retrain_log),
        'retraining_events': sum(1 for e in retrain_log
                                 if e['retrained']),
    },
    'deliverables': {
        'test_suite'      : 'tests/test_pipeline.py',
        'dashboard'       : 'dashboard/app.py',
        'adaptive_script' : 'src/adaptive_learner.py',
        'phase_reports'   : 'reports/phase*_report.json',
    }
}

with open('../reports/phase8_report.json', 'w') as f:
    json.dump(phase8_report, f, indent=2)

print("=" * 50)
print("PHASE 8 COMPLETE")
print("=" * 50)
print(json.dumps(phase8_report, indent=2))

PHASE 8 COMPLETE
{
  "phase": 8,
  "label_convention": "0.0=Phishing  |  1.0=Legitimate",
  "pytest": {
    "total_tests": 29,
    "passed": 21,
    "failed": 8,
    "errors": 0,
    "all_passed": false
  },
  "adversarial_testing": {
    "phishing_urls_tested": 5,
    "phishing_detected": 0,
    "detection_rate_pct": 0.0,
    "legitimate_urls_tested": 4,
    "legitimate_correct": 3,
    "legitimate_accuracy_pct": 75.0,
    "note": "Adversarial URLs use HTTPS and clean structure \u2014 URL features alone are insufficient to catch them. Graph features (domain_url_count) would improve this."
  },
  "final_model_performance": {
    "test_samples": 20175,
    "accuracy": 0.9958,
    "precision_macro": 0.9946,
    "recall_macro": 0.9965,
    "f1_macro": 0.9955,
    "phishing_caught": 7470,
    "phishing_missed": 6,
    "legit_cleared": 12621,
    "legit_flagged": 78,
    "phishing_miss_pct": 0.08,
    "legit_alarm_pct": 0.61
  },
  "adaptive_module": {
    "total_checks": 13,
    "retrainin

In [12]:
import pandas as pd

df = pd.read_csv('../data/raw/cicids_wednesday.csv', low_memory=False)
df.columns = df.columns.str.strip()

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nLabel distribution:")
print(df[' Label'].value_counts() if ' Label' in df.columns 
      else df['Label'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/cicids_wednesday.csv'